In [189]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [190]:
raw = Path("../PLG_SAAS_ANALYSIS/data/raw")
accounts_p = raw / "accounts.csv"
churn_events_p  = raw / "churn_events.csv"
feature_usage_p = raw / "feature_usage.csv"
subs_p = raw / "subscriptions.csv"
st_p = raw / "support_tickets.csv"

In [191]:
accounts = pd.read_csv(accounts_p)
churn_events = pd.read_csv(churn_events_p)
feature_usage = pd.read_csv(feature_usage_p)
subs = pd.read_csv(subs_p)
st = pd.read_csv(st_p)

In [192]:

accounts.head(10)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True
5,A-1b9609,Company_5,EdTech,IN,2023-10-12,ads,Enterprise,4,False,False
6,A-a0ca4e,Company_6,Cybersecurity,US,2024-03-08,ads,Pro,11,False,False
7,A-e5d6ab,Company_7,EdTech,US,2023-04-15,partner,Pro,3,False,False
8,A-7dacce,Company_8,Cybersecurity,CA,2024-09-10,event,Enterprise,12,False,True
9,A-10b8da,Company_9,DevTools,US,2023-05-08,partner,Enterprise,14,False,True


In [193]:
churn_events.isnull().sum()

churn_event_id                0
account_id                    0
churn_date                    0
reason_code                   0
refund_amount_usd             0
preceding_upgrade_flag        0
preceding_downgrade_flag      0
is_reactivation               0
feedback_text               148
dtype: int64

In [194]:
churn_events = churn_events.fillna('NaN')

In [195]:
feature_usage.isnull().sum()

usage_id               0
subscription_id        0
usage_date             0
feature_name           0
usage_count            0
usage_duration_secs    0
error_count            0
is_beta_feature        0
dtype: int64

In [196]:
subs.isnull().sum()
subs.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


In [197]:
subs['still_active'] = subs['end_date'].isna()
subs['end_date'] = subs['end_date'].fillna(pd.Timestamp('2099-01-01'))
subs.tail(25)

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,still_active
4975,S-f9a56a,A-0be015,2024-11-19,2099-01-01 00:00:00,Basic,24,456,5472,False,False,False,False,monthly,False,True
4976,S-f783b6,A-2d4502,2024-08-31,2099-01-01 00:00:00,Enterprise,7,1393,16716,False,False,False,False,monthly,True,True
4977,S-6d9705,A-726cfa,2024-03-25,2099-01-01 00:00:00,Enterprise,11,2189,26268,False,False,False,False,annual,True,True
4978,S-93ab2d,A-f17767,2023-05-02,2099-01-01 00:00:00,Basic,2,38,456,False,False,False,False,annual,True,True
4979,S-617065,A-a45270,2024-10-01,2099-01-01 00:00:00,Enterprise,3,597,7164,False,True,False,False,annual,True,True
4980,S-329803,A-1f0ac7,2024-04-29,2099-01-01 00:00:00,Basic,34,646,7752,False,False,False,False,monthly,False,True
4981,S-677801,A-80eeb6,2024-05-02,2099-01-01 00:00:00,Enterprise,89,17711,212532,False,False,False,False,monthly,True,True
4982,S-3092cb,A-6a4e2d,2023-08-03,2099-01-01 00:00:00,Enterprise,22,4378,52536,False,False,False,False,annual,True,True
4983,S-f9469c,A-781cc0,2024-09-04,2099-01-01 00:00:00,Enterprise,12,2388,28656,False,True,False,False,monthly,True,True
4984,S-c13970,A-59f724,2024-06-27,2099-01-01 00:00:00,Basic,36,684,8208,False,False,False,False,monthly,True,True


In [198]:
churn_events.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [199]:
st.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False


In [200]:
st['satisfaction_score'] = st['satisfaction_score'].fillna(st['satisfaction_score'].median())
st.head(10)

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,4.0,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,4.0,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,4.0,False
5,T-90f06d,A-94c3cd,2023-07-27,2023-07-27 09:00:00,9.0,medium,60,4.0,False
6,T-30b537,A-2e4581,2023-09-09,2023-09-10 03:00:00,27.0,urgent,64,3.0,False
7,T-60242d,A-72799b,2024-01-29,2024-01-29 12:00:00,12.0,low,56,4.0,True
8,T-7119c9,A-b179bf,2023-08-27,2023-08-27 16:00:00,16.0,urgent,154,4.0,False
9,T-b0edf2,A-7cfe77,2024-05-06,2024-05-07 04:00:00,28.0,medium,150,3.0,False


In [201]:
st_agg = st.groupby('account_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_resolution_time_hours=('resolution_time_hours', 'mean')
).reset_index()
feature_agg = feature_usage.groupby('subscription_id').agg(
    total_usage_count=('usage_count', 'sum'),
    avg_usage_duration_secs=('usage_duration_secs', 'mean'),
    total_error_count=('error_count', 'sum'),
    unique_features_used=('feature_name', 'nunique')
).reset_index()

In [202]:
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
churn_events['churn_date'] = pd.to_datetime(churn_events['churn_date'])
feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])
subs['start_date'] = pd.to_datetime(subs['start_date'])
subs['end_date'] = pd.to_datetime(subs['end_date'], errors='coerce')
st['submitted_at'] = pd.to_datetime(st['submitted_at'])
churn_events_clean = churn_events.drop_duplicates(subset=['account_id'], keep='last')
Master = subs.copy()
Master = pd.merge(Master, accounts, on='account_id', how='left')
Master = pd.merge(Master, feature_agg, on='subscription_id', how='left')
Master = pd.merge(Master, churn_events_clean, on='account_id', how='left')
Master = pd.merge(Master, st_agg, on='account_id', how='left')
Master.head(25)

,subscription_id,account_id,start_date,end_date,plan_tier_x,seats_x,mrr_amount,arr_amount,is_trial_x,upgrade_flag,...,churn_event_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text,total_tickets,avg_resolution_time_hours
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,4.0,33.250000
1,S-0f6f44,A-9b9fe9,2024-06-11,2099-01-01,Pro,17,833,9996,False,False,...,C-8e5c25,2023-12-27,features,27.92,False,False,False,too expensive,4.0,33.500000
2,S-51c0d1,A-659280,2024-11-25,2099-01-01,Enterprise,62,0,0,True,True,...,C-921335,2024-07-23,features,0.00,False,False,False,too expensive,4.0,8.000000
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,...,C-f5a863,2024-09-30,budget,6.20,False,False,True,NaN,1.0,45.000000
4,S-cff5a2,A-ba6516,2024-01-10,2099-01-01,Enterprise,27,5373,64476,False,False,...,C-b411ad,2024-03-17,budget,0.00,False,False,False,NaN,6.0,31.500000
5,S-4b9b13,A-fa2041,2024-08-13,2099-01-01,Pro,15,735,8820,False,False,...,C-21efb6,2024-07-13,competitor,0.00,False,False,False,switched to competitor,1.0,16.000000
6,S-dceac6,A-417d2f,2023-12-30,2099-01-01,Enterprise,4,796,9552,False,False,...,C-e4261a,2024-12-30,budget,0.00,False,False,False,missing features,6.0,43.500000
7,S-8cad7b,A-5f2961,2024-12-23,2099-01-01,Basic,16,304,3648,False,False,...,C-258ade,2024-11-28,pricing,6.60,False,False,False,switched to competitor,1.0,53.000000
8,S-24796e,A-cc8c8f,2024-11-27,2099-01-01,Enterprise,23,4577,54924,False,False,...,C-64ba8a,2024-09-12,budget,0.00,True,False,False,missing features,2.0,14.500000
9,S-d0c344,A-80eeb6,2024-10-27,2099-01-01,Pro,22,1078,12936,False,False,...,C-5f203c,2024-02-09,budget,0.00,False,False,False,too expensive,4.0,40.250000


In [203]:

Master.isnull().sum()

subscription_id                 0
account_id                      0
start_date                      0
end_date                        0
plan_tier_x                     0
seats_x                         0
mrr_amount                      0
arr_amount                      0
is_trial_x                      0
upgrade_flag                    0
downgrade_flag                  0
churn_flag_x                    0
billing_frequency               0
auto_renew_flag                 0
still_active                    0
account_name                    0
industry                        0
country                         0
signup_date                     0
referral_source                 0
plan_tier_y                     0
seats_y                         0
is_trial_y                      0
churn_flag_y                    0
total_usage_count              33
avg_usage_duration_secs        33
total_error_count              33
unique_features_used           33
churn_event_id               1472
churn_date    

In [204]:
Master[['churn_flag_x', 'churn_flag_y']].head(10)

,churn_flag_x,churn_flag_y
0,True,False
1,False,False
2,False,True
3,True,False
4,False,False
5,False,False
6,False,False
7,False,False
8,False,False
9,False,False


In [205]:
print("Duplicates in subs (subscription_id):", subs.duplicated(subset=['subscription_id']).sum())
print("Duplicates in subs (account_id):", subs.duplicated(subset=['account_id']).sum())
print("Duplicates in accounts (account_id):", accounts.duplicated(subset=['account_id']).sum())
print("Duplicates in st_agg (account_id):", st_agg.duplicated(subset=['account_id']).sum())
print("Duplicates in feature_agg (subscription_id):", feature_agg.duplicated(subset=['subscription_id']).sum())


Duplicates in subs (subscription_id): 0
Duplicates in subs (account_id): 4500
Duplicates in accounts (account_id): 0
Duplicates in st_agg (account_id): 0
Duplicates in feature_agg (subscription_id): 0


In [206]:
Master['churn_date'] = Master['churn_date'].fillna(pd.Timestamp('2099-01-01'))
Master['churn_date'].isnull().sum()

np.int64(0)

In [232]:
Master['feedback_text'] = Master['feedback_text'].fillna('Nan')
Master['is_reactivation'] = Master['is_reactivation'].fillna('Nan')
Master['avg_usage_duration_secs'] = Master['avg_usage_duration_secs'].fillna(Master['avg_usage_duration_secs'].median())
Master['total_usage_count'] = Master['total_usage_count'].fillna(Master['total_usage_count'].median())
Master['preceding_upgrade_flag'] = Master['preceding_upgrade_flag'].fillna('Nan')
Master['reason_code'] = Master['reason_code'].fillna('Nan')
Master['churn_event_id'] = Master['churn_event_id'].fillna('Nan')
Master['refund_amount_usd'] = Master['refund_amount_usd'].fillna(Master['refund_amount_usd'].median())
Master['total_error_count'] = Master['total_error_count'].fillna(Master['total_error_count'].median())
Master['unique_features_used'] = Master['unique_features_used'].fillna(Master['unique_features_used'].median())
Master['preceding_downgrade_flag'] = Master['preceding_downgrade_flag'].fillna('Nan')
Master['total_tickets'] = Master['total_tickets'].fillna(Master['total_tickets'].median())
Master['avg_resolution_time_hours'] = Master['avg_resolution_time_hours'].fillna(Master['avg_resolution_time_hours'].median())

In [233]:
Master.head()

,subscription_id,account_id,start_date,end_date,plan_tier_x,seats_x,mrr_amount,arr_amount,is_trial_x,upgrade_flag,...,churn_event_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text,total_tickets,avg_resolution_time_hours
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,...,Nan,2099-01-01,Nan,0.00,Nan,Nan,Nan,Nan,4.0,33.25
1,S-0f6f44,A-9b9fe9,2024-06-11,2099-01-01,Pro,17,833,9996,False,False,...,C-8e5c25,2023-12-27,features,27.92,False,False,False,too expensive,4.0,33.50
2,S-51c0d1,A-659280,2024-11-25,2099-01-01,Enterprise,62,0,0,True,True,...,C-921335,2024-07-23,features,0.00,False,False,False,too expensive,4.0,8.00
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,...,C-f5a863,2024-09-30,budget,6.20,False,False,True,NaN,1.0,45.00
4,S-cff5a2,A-ba6516,2024-01-10,2099-01-01,Enterprise,27,5373,64476,False,False,...,C-b411ad,2024-03-17,budget,0.00,False,False,False,NaN,6.0,31.50


In [234]:
Master.isnull().sum()

subscription_id              0
account_id                   0
start_date                   0
end_date                     0
plan_tier_x                  0
seats_x                      0
mrr_amount                   0
arr_amount                   0
is_trial_x                   0
upgrade_flag                 0
downgrade_flag               0
churn_flag_x                 0
billing_frequency            0
auto_renew_flag              0
still_active                 0
account_name                 0
industry                     0
country                      0
signup_date                  0
referral_source              0
plan_tier_y                  0
seats_y                      0
is_trial_y                   0
churn_flag_y                 0
total_usage_count            0
avg_usage_duration_secs      0
total_error_count            0
unique_features_used         0
churn_event_id               0
churn_date                   0
reason_code                  0
refund_amount_usd            0
precedin

In [ ]:

processed_dir.mkdir(parents=True, exist_ok=True)
Master.to_csv(processed_dir / "master_table.csv", index=False)